# 03 — Gold Layer Analytics Tables & SQL KPIs (Databricks Spark SQL, Delta Lake, PyArrow & DuckDB)
- **Workstream 2:** Lakehouse & Data Quality (Member 2 — Ahmed Essam)
- **ClickUp Task:** `123u8tync4w` (*Build Gold analytics-ready tables*)
- **Delta Tables Created:**
  1. `workspace.default.gold_pharma_catalog_analytics`
  2. `workspace.default.gold_recall_supply_risk_kpis`

In [ ]:
# 1. Gold Table 1: Active Ingredient Supply Chain Diversity, Single-Supplier Risk & WHO EML Match
gold_catalog_df = spark.sql("""
    SELECT
        primary_active_ingredient AS active_ingredient,
        CAST(COUNT(DISTINCT product_id) AS INT) AS total_products,
        CAST(COUNT(DISTINCT labeler_name) AS INT) AS distinct_labelers_count,
        CASE WHEN COUNT(DISTINCT labeler_name) = 1 THEN TRUE ELSE FALSE END AS single_supplier_risk,
        array_join(array_sort(collect_set(dosage_form)), ', ') AS dosage_forms,
        array_join(array_sort(collect_set(marketing_category)), ', ') AS marketing_categories,
        MAX(is_who_essential_ingredient) AS is_who_essential,
        'VERIFIED_REGULATORY_SNAPSHOT_NOT_LIVE_STOCK' AS data_scope_label
    FROM workspace.default.silver_openfda_ndc
    GROUP BY primary_active_ingredient
    ORDER BY total_products DESC, active_ingredient ASC
""")

(
    gold_catalog_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.gold_pharma_catalog_analytics")
)
print(f"gold_pharma_catalog_analytics rows: {gold_catalog_df.count()}")
display(gold_catalog_df.limit(15))

In [ ]:
# 2. Gold Table 2: Drug Recall Supply-Chain Risk & Root-Cause KPIs (Spark SQL + Delta Lake)
gold_recall_df = spark.sql("""
    SELECT
        CONCAT(UPPER(REPLACE(classification, ' ', '_')), '__', recall_reason_category, '__', REPLACE(country, ' ', '_')) AS risk_segment_id,
        classification,
        CAST(MIN(severity_rank) AS INT) AS severity_rank,
        recall_reason_category,
        country,
        CAST(COUNT(*) AS INT) AS total_recalls,
        CAST(SUM(CASE WHEN UPPER(status) = 'ONGOING' THEN 1 ELSE 0 END) AS INT) AS ongoing_recalls,
        CAST(SUM(CASE WHEN UPPER(status) IN ('TERMINATED', 'COMPLETED') THEN 1 ELSE 0 END) AS INT) AS terminated_or_completed_recalls,
        CAST(COUNT(DISTINCT recalling_firm) AS INT) AS affected_firms_count,
        'VERIFIED_US_FDA_ENFORCEMENT_SNAPSHOT' AS data_scope_label
    FROM workspace.default.silver_openfda_enforcement
    GROUP BY classification, recall_reason_category, country
    ORDER BY severity_rank ASC, total_recalls DESC
""")

(
    gold_recall_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.gold_recall_supply_risk_kpis")
)
print(f"gold_recall_supply_risk_kpis rows: {gold_recall_df.count()}")
display(gold_recall_df)

In [ ]:
# 3. PyArrow + DuckDB KPI Analytics over Gold Tables
import duckdb
import pyarrow as pa

# Convert Spark DataFrames to Arrow Tables for zero-copy DuckDB analytics
catalog_arrow = pa.Table.from_pandas(gold_catalog_df.toPandas(), preserve_index=False)
recall_arrow = pa.Table.from_pandas(gold_recall_df.toPandas(), preserve_index=False)

con = duckdb.connect(database=":memory:")
con.register("gold_catalog", catalog_arrow)
con.register("gold_recalls", recall_arrow)

kpi_summary = con.execute("""
    SELECT
        COUNT(*) AS distinct_active_ingredients,
        SUM(total_products) AS total_validated_products,
        SUM(CASE WHEN single_supplier_risk THEN 1 ELSE 0 END) AS single_supplier_ingredients,
        ROUND(100.0 * SUM(CASE WHEN single_supplier_risk THEN 1 ELSE 0 END) / COUNT(*), 2) AS single_supplier_risk_pct,
        SUM(CASE WHEN is_who_essential THEN 1 ELSE 0 END) AS who_essential_matched_ingredients
    FROM gold_catalog
""").df()

recall_kpis = con.execute("""
    SELECT
        classification,
        recall_reason_category,
        SUM(total_recalls) AS total_recalls,
        SUM(ongoing_recalls) AS ongoing_recalls
    FROM gold_recalls
    GROUP BY classification, recall_reason_category
    ORDER BY total_recalls DESC
""").df()

con.close()
display(kpi_summary)
display(recall_kpis)